# Corpus embeddings — the Colab half of the predictor

This notebook is the **only** part of the project that needs torch or a GPU. It reads
the corpus images, runs frozen encoders over them, and writes one `.npz` per encoder
that the laptop reads back with `adml.embeddings.EmbeddingSet.load_npz`.

Nothing is trained here. The trainable head is numpy and lives in
`packages/ml/adml/predictor.py`, so the model in the evaluation tables is the same
object the API serves — there is no second implementation to disagree with the first.

**The contract.** Each `.npz` holds one array per `item_id`, plus a `__spec__` entry
recording which model produced it. A file without `__spec__` is refused on load: an
embedding block whose provenance is unknown cannot appear in a reported result.

**What to bring:**

1. `corpus.zip` — the generated images.
   `cd fixtures && zip -qr /tmp/corpus.zip corpus/`
2. `manifest.json` — written by `scripts/extract_features.py`. It maps `item_id` to the
   storage key, which is what stops a filename mismatch from silently dropping items.

**Runtime:** Runtime → Change runtime type → T4 GPU. On free Colab, 300 images take
about a minute per encoder.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU — set Runtime > T4"
!pip install -q "transformers>=4.44" "torch>=2.3" pillow numpy

In [ ]:
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# float16 on GPU halves the encoder's memory and changes embeddings in the fourth
# decimal place; the head standardises its inputs, so that is well below anything
# the model can resolve.
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"device {DEVICE}, dtype {DTYPE}")

OUT = Path("embeddings")
OUT.mkdir(exist_ok=True)

## 2. Upload the corpus and the manifest

Run the cell, then pick `corpus.zip` and `manifest.json` together.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
            archive.extractall("corpus_root")
        print(f"extracted {name}")

manifest = json.loads(uploaded["manifest.json"])
print(f"manifest: {len(manifest['items'])} {manifest['kind']} items")

In [ ]:
def resolve(key: str) -> Path | None:
    """Find an item's file inside the extracted zip.

    Matched on the storage key's trailing path rather than on the basename alone:
    every set writes `i0.png`, `i1.png`, `i2.png`, so basenames collide across sets
    and matching on them would silently pair an item with another set's image.
    """
    root = Path("corpus_root")
    direct = root / key
    if direct.exists():
        return direct
    tail = Path(key)
    for candidate in root.rglob(tail.name):
        if str(candidate).endswith(str(Path(*tail.parts[-3:]))):
            return candidate
    return None


ITEMS = []
missing = []
for entry in manifest["items"]:
    path = resolve(entry["key"])
    (ITEMS if path else missing).append((entry["item_id"], path) if path else entry["item_id"])

print(f"resolved {len(ITEMS)} images, {len(missing)} missing")
if missing:
    print("MISSING (these items will be dropped from training, not zero-filled):")
    print("  " + ", ".join(missing[:10]))

## 3. The write contract

`save_npz` here matches `adml.embeddings.EmbeddingSet.save_npz` byte for byte, so the
laptop needs no special handling for files this notebook produced. Kept inline rather
than imported so the notebook runs without the repo checked out in Colab.

In [ ]:
def save_npz(path, name, source, vectors, l2_normalised=False):
    dims = {v.shape[0] for v in vectors.values()}
    assert len(dims) == 1, f"inconsistent widths: {dims}"
    spec = json.dumps(
        {
            "name": name,
            "dim": int(dims.pop()),
            "source": source,
            # True: a real encoder ran. `hash_embeddings` is the only thing that writes
            # False, and that flag travels into every table and every reported number.
            "is_real": True,
            "l2_normalised": bool(l2_normalised),
        }
    )
    payload = {k: v.astype(np.float32) for k, v in vectors.items()}
    payload["__spec__"] = np.frombuffer(spec.encode(), dtype=np.uint8)
    np.savez_compressed(path, **payload)
    print(f"wrote {path}  {len(vectors)} items x {spec}")


@torch.no_grad()
def encode_all(embed_batch, batch_size=16):
    """Run `embed_batch` over every resolved item, in order."""
    out = {}
    for start in range(0, len(ITEMS), batch_size):
        chunk = ITEMS[start : start + batch_size]
        images = [Image.open(p).convert("RGB") for _, p in chunk]
        vectors = embed_batch(images).float().cpu().numpy()
        for (item_id, _), vec in zip(chunk, vectors, strict=True):
            out[item_id] = vec
        if start % (batch_size * 8) == 0:
            print(f"  {start + len(chunk)}/{len(ITEMS)}")
    return out

## 4. SigLIP — general visual semantics

`siglip-base-patch16-224` rather than a large variant. The head has on the order of a
thousand pairwise training comparisons, which supports roughly eighty free parameters
(`adml.featureset.parameter_budget`); the block is reduced to 8 principal components
per fold regardless of whether it arrives as 768 dimensions or 1152. Paying for the
larger encoder buys nothing that survives that reduction.

In [ ]:
from transformers import SiglipModel, SiglipProcessor

SIGLIP_ID = "google/siglip-base-patch16-224"
siglip = SiglipModel.from_pretrained(SIGLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()
siglip_processor = SiglipProcessor.from_pretrained(SIGLIP_ID)


def siglip_batch(images):
    inputs = siglip_processor(images=images, return_tensors="pt").to(DEVICE, DTYPE)
    return siglip.get_image_features(**inputs)


siglip_vectors = encode_all(siglip_batch)
save_npz(OUT / "siglip.npz", "siglip", SIGLIP_ID, siglip_vectors)
del siglip
torch.cuda.empty_cache()

## 5. DINOv2 — appearance and layout

Complementary to SigLIP rather than redundant with it: SigLIP is trained against text
and captures *what the image is about*, while DINOv2 is self-supervised on pixels and
captures *how it looks*. Whether that distinction earns its place is an ablation row,
not an assumption — `scripts/train_predictor.py` reports one line per feature group.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINO_ID = "facebook/dinov2-base"
dino = AutoModel.from_pretrained(DINO_ID, torch_dtype=DTYPE).to(DEVICE).eval()
dino_processor = AutoImageProcessor.from_pretrained(DINO_ID)


def dino_batch(images):
    inputs = dino_processor(images=images, return_tensors="pt").to(DEVICE, DTYPE)
    # The CLS token, not the mean of the patch tokens: DINOv2's CLS is what the
    # self-distillation objective actually shapes into a global descriptor.
    return dino(**inputs).last_hidden_state[:, 0]


dino_vectors = encode_all(dino_batch)
save_npz(OUT / "dinov2.npz", "dinov2", DINO_ID, dino_vectors)
del dino
torch.cuda.empty_cache()

## 6. Aesthetic score — optional

The LAION aesthetic predictor is a small head over CLIP ViT-L/14 embeddings, and its
weights come from a third-party URL rather than from the Hugging Face hub. Run this
only if you are willing to depend on that; the pipeline is complete without it, and
the ablation table will show the aesthetic row as unavailable rather than break.

It lands in the feature table as `FeatureGroup.AESTHETIC` — one column, not a block —
because it is a scalar judgement of quality and reducing it further makes no sense.

In [ ]:
RUN_AESTHETIC = False  # set True to fetch the third-party head

if RUN_AESTHETIC:
    import urllib.request

    import torch.nn as nn
    from transformers import CLIPModel, CLIPProcessor

    CLIP_ID = "openai/clip-vit-large-patch14"
    clip = CLIPModel.from_pretrained(CLIP_ID, torch_dtype=torch.float32).to(DEVICE).eval()
    clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

    WEIGHTS = (
        "https://github.com/christophschuhmann/improved-aesthetic-predictor/"
        "raw/main/sac+logos+ava1-l14-linearMSE.pth"
    )
    urllib.request.urlretrieve(WEIGHTS, "aesthetic.pth")

    head = nn.Sequential(
        nn.Linear(768, 1024),
        nn.Dropout(0.2),
        nn.Linear(1024, 128),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.Dropout(0.1),
        nn.Linear(64, 16),
        nn.Linear(16, 1),
    )
    head.load_state_dict(torch.load("aesthetic.pth", map_location="cpu"))
    head.to(DEVICE).eval()

    def aesthetic_batch(images):
        inputs = clip_processor(images=images, return_tensors="pt").to(DEVICE)
        feats = clip.get_image_features(**inputs)
        # The published head expects L2-normalised CLIP features.
        feats = feats / feats.norm(dim=-1, keepdim=True)
        return head(feats)

    save_npz(
        OUT / "aesthetic.npz",
        "aesthetic",
        "laion/improved-aesthetic-predictor-l14",
        encode_all(aesthetic_batch, batch_size=8),
    )
    del clip, head
    torch.cuda.empty_cache()
else:
    print("skipped — the pipeline is complete without it")

## 7. Verify before downloading

Reads each file back and checks the three things that would otherwise fail silently on
the laptop: the spec is present, every item is covered, and no vector is degenerate.

In [ ]:
expected = {item_id for item_id, _ in ITEMS}
ok = True
for path in sorted(OUT.glob("*.npz")):
    with np.load(path) as data:
        assert "__spec__" in data.files, f"{path} has no provenance"
        spec = json.loads(bytes(data["__spec__"]).decode())
        keys = {k for k in data.files if k != "__spec__"}
        rows = np.stack([data[k] for k in sorted(keys)])
    absent = expected - keys
    norms = np.linalg.norm(rows, axis=1)
    dead = int((norms < 1e-8).sum())
    constant = bool(np.allclose(rows.std(axis=0), 0.0))
    print(
        f"{path.name:<16} {spec['name']:<10} dim {spec['dim']:>4}  items {len(keys):>4}  "
        f"missing {len(absent):>3}  zero-vectors {dead}  all-identical {constant}"
    )
    ok = ok and not absent and not dead and not constant
print("\nOK" if ok else "\nPROBLEMS ABOVE — do not train on these")

In [ ]:
!zip -qr embeddings.zip embeddings/
files.download("embeddings.zip")

## 8. Back on the laptop

```bash
unzip -o ~/Downloads/embeddings.zip -d fixtures/corpus/
.venv/bin/python scripts/train_predictor.py --embeddings fixtures/corpus/embeddings
```

The training script prints the embedding blocks it found and names the ones still
absent, so a partial run is a smaller experiment rather than a silent one.

### Stage A pretraining is not here

The plan's Stage A — pretrain on SMPD-Video popularity and Pitt Ads, then calibrate on
pairwise judgements — is a separate notebook, and it is **blocked on data access rather
than on code**: both datasets need registration, and SMPD is large enough to matter for
Colab's disk. Stage B (the pairwise calibration) is what
`scripts/train_predictor.py` already runs, and it does not depend on Stage A. Treat the
Stage A row of the ablation table as pending, not as zero.